# MTL QSAR Cardiotoxicity - Regression (Revised)
## Union SMILES + Task-Specific Backbone
> **Perubahan utama vs versi sebelumnya:**
> - Dataset: **UNION** semua SMILES (tidak ada data terbuang)
> - Arsitektur: **Shared Encoder** + **Task-Specific Backbone** per endpoint
> - Semua fitur output (log, plot, CSV, Excel, artefak) dipertahankan

In [1]:
import os, sys, json, ast, copy, random, pickle, logging, csv, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore', category=UserWarning)

# -- GLOBAL SETTINGS --
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
KEY_COL       = 'SMILES'
LABEL_COL     = 'pIC50'
OUT_ROOT      = r'D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED'
NUM_WORKERS   = 0
PIN_MEMORY    = (DEVICE == 'cuda')
NON_BLOCKING  = True
CV_FOLDS      = 10
DROPOUT       = 0.2
ALL_ENDPOINTS = ['hERG', 'Cav1.2', 'Nav1.5']
GRAD_CLIP     = 5.0

EPOCHS_MAP   = {'hERG': 65, 'Cav1.2': 40, 'Nav1.5': 40}
PATIENCE_MAP = {'hERG': 15, 'Cav1.2': 15,  'Nav1.5': 15}
MAX_EPOCHS_MTL = 75
PATIENCE_MTL   = 15

os.makedirs(OUT_ROOT, exist_ok=True)
print(f'[INFO] DEVICE={DEVICE} | OUT_ROOT={OUT_ROOT}')


[INFO] DEVICE=cuda | OUT_ROOT=D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED


c:\Users\USER-PC\anaconda3\envs\torch-train\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FP_PATHS = {
    'Nav1.5': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\Nav1.5_DevSet_with_ECFP4_MACCS_APF.csv',
    'Cav1.2': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\Cav1.2_DevSet_with_ECFP4_MACCS_APF.csv',
    'hERG':   r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\hERG_DevSet_with_ECFP4_MACCS_APF.csv',
}
GRAPH_PATHS = {
    'Nav1.5': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Nav1.5_DevSet_graphs_67feat_with_pIC50.pkl',
    'Cav1.2': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Cav1.2_DevSet_graphs_67feat_with_pIC50.pkl',
    'hERG':   r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\hERG_DevSet_graphs_67feat_with_pIC50.pkl',
}
MORDRED_PATHS = {
    'Nav1.5': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\PP\Nav1.5_DevSet_RDKit_CDK.xlsx',
    'Cav1.2': r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\PP\Cav1.2_DevSet_RDKit_CDK.xlsx',
    'hERG':   r'D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\PP\hERG_DevSet_RDKit_CDK.xlsx',
}
META_MD_COLS = [
    'SMILES', 'Activity', 'pIC50', 'Source',
    'Activity_Label', 'Compound_ID', 'InChIKey',
    'Max_Tanimoto_to_Dev', 'MaxTanimototoDev',
]


In [3]:
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark     = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

def fast_parse_list(s):
    if isinstance(s, list):       return s
    if isinstance(s, np.ndarray): return s.tolist()
    if not isinstance(s, str):    raise TypeError(f'Unexpected type: {type(s)}')
    try:    return json.loads(s)
    except: return ast.literal_eval(s)

def compute_fp_matrix_from_df(df_fp, ec_col='ECFP4', mc_col='MACCS'):
    ec = df_fp[ec_col].tolist(); mc = df_fp[mc_col].tolist()
    ec0 = fast_parse_list(ec[0]); mc0 = fast_parse_list(mc[0])
    d_ec, d_mc = len(ec0), len(mc0)
    X = np.zeros((len(df_fp), d_ec + d_mc), dtype=np.float32)
    for i in range(len(df_fp)):
        e = np.asarray(fast_parse_list(ec[i]), dtype=np.float32)
        m = np.asarray(fast_parse_list(mc[i]), dtype=np.float32)
        X[i, :d_ec] = e; X[i, d_ec:] = m
    return X, d_ec, d_mc

def get_mordred_feature_cols(df_md, meta_cols):
    return [c for c in df_md.columns
            if c not in meta_cols and pd.api.types.is_numeric_dtype(df_md[c])]

def compute_regression_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    mse  = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae  = float(mean_absolute_error(y_true, y_pred))
    denom = np.maximum(np.abs(y_true), eps)
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    r2   = float(r2_score(y_true, y_pred))
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'mape': mape, 'r2': r2}

class TargetScaler:
    '''Per-endpoint z-score scaler untuk pIC50.'''
    def __init__(self):
        self.mean_ = {}; self.std_ = {}
    def fit(self, ep, y):
        self.mean_[ep] = float(np.nanmean(y))
        self.std_[ep]  = float(np.nanstd(y)) + 1e-8
        return self
    def transform(self, ep, y):
        return (np.asarray(y, dtype=np.float32) - self.mean_[ep]) / self.std_[ep]
    def inverse_transform(self, ep, y):
        return np.asarray(y, dtype=np.float32) * self.std_[ep] + self.mean_[ep]
    def fit_transform(self, ep, y):
        return self.fit(ep, y).transform(ep, y)


## Section 1 - Data Loading & Mordred Alignment

In [4]:
def align_mordred_cols(raw_mordred_dict, meta_cols=META_MD_COLS, strategy='intersection'):
    feat_cols_per_ep = {}
    for ep, df in raw_mordred_dict.items():
        feat_cols_per_ep[ep] = set(get_mordred_feature_cols(df, meta_cols))
        print(f'[MD Align] {ep}: {len(feat_cols_per_ep[ep])} feature cols')
    if strategy == 'intersection':
        shared = set.intersection(*feat_cols_per_ep.values())
        shared_feat_cols = sorted(list(shared))
        for ep, cols in feat_cols_per_ep.items():
            dropped = cols - shared
            if dropped: print(f'  WARNING {ep}: {len(dropped)} cols DROPPED')
    else:
        shared_feat_cols = sorted(list(set.union(*feat_cols_per_ep.values())))
    print(f'[MD Align] {strategy.upper()} -> {len(shared_feat_cols)} shared feature cols')
    aligned_dict = {}
    for ep, df in raw_mordred_dict.items():
        meta_present = [c for c in meta_cols if c in df.columns]
        df_aligned   = df[meta_present].copy()
        for col in shared_feat_cols:
            df_aligned[col] = df[col].values if col in df.columns else 0.0
        df_aligned[shared_feat_cols] = (
            df_aligned[shared_feat_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0.0)
        )
        aligned_dict[ep] = df_aligned
    log_path = os.path.join(OUT_ROOT, 'md_aligned_feature_cols.txt')
    with open(log_path, 'w') as f:
        f.write(f'Strategy: {strategy}\nN features: {len(shared_feat_cols)}\n\n')
        for col in shared_feat_cols: f.write(col + '\n')
    print(f'[MD Align] Saved -> {log_path}')
    return aligned_dict, shared_feat_cols

RAW_FP = {}; RAW_MORDRED = {}; RAW_GRAPH = {}
for ep in ALL_ENDPOINTS:
    RAW_FP[ep]      = pd.read_csv(FP_PATHS[ep])
    RAW_MORDRED[ep] = pd.read_excel(MORDRED_PATHS[ep])
    g_path = GRAPH_PATHS[ep]
    if os.path.exists(g_path):
        with open(g_path, 'rb') as f: RAW_GRAPH[ep] = pickle.load(f)
    else:
        RAW_GRAPH[ep] = None
    print(f'[LOAD] {ep}: FP={RAW_FP[ep].shape}, MD={RAW_MORDRED[ep].shape}, '
          f"Graph={'OK' if RAW_GRAPH[ep] else 'MISSING'}")

RAW_MORDRED_ALIGNED, SHARED_MD_COLS = align_mordred_cols(RAW_MORDRED)
print(f'[INFO] MD feat dim after alignment: {len(SHARED_MD_COLS)}')


[LOAD] hERG: FP=(20020, 11), MD=(20020, 59), Graph=OK
[LOAD] Cav1.2: FP=(1138, 11), MD=(1138, 59), Graph=OK
[LOAD] Nav1.5: FP=(4599, 11), MD=(4599, 59), Graph=OK
[MD Align] hERG: 51 feature cols
[MD Align] Cav1.2: 51 feature cols
[MD Align] Nav1.5: 51 feature cols
[MD Align] INTERSECTION -> 51 shared feature cols
[MD Align] Saved -> D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\md_aligned_feature_cols.txt
[INFO] MD feat dim after alignment: 51


## Section 2 - Build CORE_UNI & CORE_MULTI per endpoint

In [5]:
CORE_UNI   = {}
CORE_MULTI = {}

for ep in ALL_ENDPOINTS:
    print(f'\n### Building CORE for {ep} ###')
    df_fp = RAW_FP[ep]
    df_md = RAW_MORDRED_ALIGNED[ep]
    g_obj = RAW_GRAPH[ep]

    X_fp_full, d_ec, d_mc = compute_fp_matrix_from_df(df_fp)
    y_fp_full      = df_fp[LABEL_COL].astype(np.float32).to_numpy()
    smiles_fp_full = df_fp[KEY_COL].astype(str).tolist()

    md_cols        = [c for c in SHARED_MD_COLS if c in df_md.columns]
    X_md_full      = df_md[md_cols].to_numpy(dtype=np.float32)
    y_md_full      = df_md[LABEL_COL].astype(np.float32).to_numpy()
    smiles_md_full = df_md[KEY_COL].astype(str).tolist()

    if g_obj is not None:
        graphs_full = []; y_g_full = []; smiles_g_full = []
        for item in g_obj:
            gd   = item['graph']
            x_np = np.asarray(gd['node_features'], dtype=np.float32)
            ei   = np.asarray(gd['edge_index'],    dtype=np.int64)
            if ei.ndim == 2 and ei.shape[0] != 2: ei = ei.T
            graphs_full.append(Data(x=torch.from_numpy(x_np),
                                    edge_index=torch.from_numpy(ei)))
            y_g_full.append(float(item.get(LABEL_COL, item.get('pIC50', 0.0))))
            smiles_g_full.append(item['SMILES'])
        y_g_full = np.array(y_g_full, dtype=np.float32)
    else:
        graphs_full = y_g_full = smiles_g_full = None

    CORE_UNI[ep] = {
        'fp': {'X': X_fp_full, 'y': y_fp_full, 'smiles': smiles_fp_full,
               'd_ec': d_ec, 'd_mc': d_mc},
        'md': {'X': X_md_full, 'y': y_md_full, 'smiles': smiles_md_full, 'cols': md_cols},
        'g':  {'graphs': graphs_full, 'y': y_g_full, 'smiles': smiles_g_full},
    }

    s_fp = set(smiles_fp_full)
    s_md = set(smiles_md_full)
    s_g  = set(smiles_g_full) if smiles_g_full is not None else None
    smiles_multi = sorted(list(s_fp & s_md & s_g) if s_g else list(s_fp & s_md))

    idx_fp = {s: i for i, s in enumerate(smiles_fp_full)}
    idx_md = {s: i for i, s in enumerate(smiles_md_full)}
    idx_g  = {s: i for i, s in enumerate(smiles_g_full)} if smiles_g_full else {}

    fp_arr = X_fp_full[np.array([idx_fp[s] for s in smiles_multi])]
    md_arr = X_md_full[np.array([idx_md[s] for s in smiles_multi])]
    y_arr  = y_fp_full[np.array([idx_fp[s] for s in smiles_multi])]
    gr_arr = [graphs_full[idx_g[s]] for s in smiles_multi] if graphs_full else None

    if smiles_g_full is not None and gr_arr is not None:
        y_g_arr  = y_g_full[np.array([idx_g[s] for s in smiles_multi])]
        y_md_arr = y_md_full[np.array([idx_md[s] for s in smiles_multi])]
        keep = (np.abs(y_arr - y_md_arr) <= 1e-5) & (np.abs(y_arr - y_g_arr) <= 1e-5)
        n_before = len(smiles_multi)
        smiles_multi = [s for s, k in zip(smiles_multi, keep) if k]
        fp_arr = fp_arr[keep]; md_arr = md_arr[keep]; y_arr = y_arr[keep]
        gr_arr = [g for g, k in zip(gr_arr, keep) if k]
        print(f'    [F4] Label consistency: {n_before} -> {len(smiles_multi)} compounds')

    CORE_MULTI[ep] = {
        'X_fp': fp_arr, 'X_md': md_arr, 'graphs': gr_arr,
        'y': y_arr, 'smiles': smiles_multi,
    }
    print(f'    Multimodal size: {len(smiles_multi)}, '
          f'y range: {y_arr.min():.3f}-{y_arr.max():.3f}')

print('\nCORE_UNI & CORE_MULTI built for all endpoints.')



### Building CORE for hERG ###
    [F4] Label consistency: 20020 -> 20020 compounds
    Multimodal size: 20020, y range: 0.000-12.000

### Building CORE for Cav1.2 ###
    [F4] Label consistency: 1138 -> 1138 compounds
    Multimodal size: 1138, y range: 1.800-11.097

### Building CORE for Nav1.5 ###
    [F4] Label consistency: 4599 -> 4599 compounds
    Multimodal size: 4599, y range: 2.300-9.553

CORE_UNI & CORE_MULTI built for all endpoints.


## Section 3 - Joint MTL Dataset (UNION SMILES)
> **REVISI UTAMA:** Dataset dibangun dari **UNION** semua SMILES, bukan intersection.
> Senyawa yang tidak ada di suatu endpoint diberi NaN dan di-mask saat loss computation.
> **Tidak ada data yang terbuang.**

In [6]:
ENDPOINT_ORDER = ['hERG', 'Cav1.2', 'Nav1.5']

def build_mtl_joint_dataset(core_multi_dict, endpoint_order=ENDPOINT_ORDER,
                              use_fp=True, use_md=True, use_g=False):
    '''
    REVISED: Dataset dari UNION semua SMILES.
    Senyawa yang tidak ada di suatu endpoint -> NaN di Y_mtl.
    Loss function handle NaN dengan mask per task.
    Tidak ada data yang terbuang.
    '''
    smiles_sets = [set(core_multi_dict[ep]['smiles']) for ep in endpoint_order]
    smiles_all  = sorted(list(set.union(*smiles_sets)))
    N = len(smiles_all)
    smiles_to_i = {s: i for i, s in enumerate(smiles_all)}

    print(f'[MTL-Union] Total SMILES (union): {N}')
    per_ep = {ep: len(core_multi_dict[ep]['smiles']) for ep in endpoint_order}
    print(f'  Per endpoint: {per_ep}')

    n_ep  = len(endpoint_order)
    Y_mtl = np.full((N, n_ep), fill_value=np.nan, dtype=np.float32)

    ep_idx_maps = {}
    for j, ep in enumerate(endpoint_order):
        idx_map = {s: i for i, s in enumerate(core_multi_dict[ep]['smiles'])}
        ep_idx_maps[ep] = idx_map
        y_ep = core_multi_dict[ep]['y']
        for smi in smiles_all:
            if smi in idx_map:
                Y_mtl[smiles_to_i[smi], j] = float(y_ep[idx_map[smi]])

    for j, ep in enumerate(endpoint_order):
        n_valid = (~np.isnan(Y_mtl[:, j])).sum()
        print(f'  {ep}: {n_valid}/{N} labeled ({100*n_valid/N:.1f}%)')

    X_fp_out = X_md_out = None
    graphs_out = None

    if use_fp:
        d_fp = core_multi_dict[endpoint_order[0]]['X_fp'].shape[1]
        X_fp_out = np.zeros((N, d_fp), dtype=np.float32)
        for i, smi in enumerate(smiles_all):
            for ep in endpoint_order:
                if smi in ep_idx_maps[ep]:
                    X_fp_out[i] = core_multi_dict[ep]['X_fp'][ep_idx_maps[ep][smi]]
                    break

    if use_md:
        d_md = core_multi_dict[endpoint_order[0]]['X_md'].shape[1]
        X_md_out = np.zeros((N, d_md), dtype=np.float32)
        for i, smi in enumerate(smiles_all):
            for ep in endpoint_order:
                if smi in ep_idx_maps[ep]:
                    X_md_out[i] = core_multi_dict[ep]['X_md'][ep_idx_maps[ep][smi]]
                    break

    if use_g:
        graphs_out = [None] * N
        for i, smi in enumerate(smiles_all):
            for ep in endpoint_order:
                if (smi in ep_idx_maps[ep]
                        and core_multi_dict[ep]['graphs'] is not None):
                    graphs_out[i] = core_multi_dict[ep]['graphs'][ep_idx_maps[ep][smi]]
                    break

    total_nan   = int(np.isnan(Y_mtl).sum())
    total_cells = N * n_ep
    print(f'[MTL-Union] Y_mtl NaN: {total_nan}/{total_cells} '
          f'({100*total_nan/total_cells:.1f}% missing - expected for union)')

    return {
        'smiles':        smiles_all,
        'X_fp':          X_fp_out,
        'X_md':          X_md_out,
        'graphs':        graphs_out,
        'Y_mtl':         Y_mtl,
        'endpoint_cols': endpoint_order,
    }

MTL_DATA = build_mtl_joint_dataset(CORE_MULTI, use_fp=True, use_md=True, use_g=True)


[MTL-Union] Total SMILES (union): 24880
  Per endpoint: {'hERG': 20020, 'Cav1.2': 1138, 'Nav1.5': 4599}
  hERG: 20020/24880 labeled (80.5%)
  Cav1.2: 1138/24880 labeled (4.6%)
  Nav1.5: 4599/24880 labeled (18.5%)
[MTL-Union] Y_mtl NaN: 48883/74640 (65.5% missing - expected for union)


## Section 4 - Model Classes
> **REVISI UTAMA:** `MTLModularNetReg` kini menggunakan:
> - **Shared Encoder** (FP/MD/Graph) -> transfer learning representasi molekul
> - **Task-Specific Backbone** per endpoint -> tidak ada konflik gradien antar task
> - **Task-Specific Head** per endpoint -> prediksi independen

In [7]:
try:
    from torch_geometric.data import Batch
    from torch_geometric.nn import GCNConv
    HAS_PYG = True
except ImportError:
    Batch = GCNConv = None
    HAS_PYG = False

# -- Dataset --
class MTLQSARDatasetReg(Dataset):
    def __init__(self, X_fp, X_md, graphs, Y):
        self.X_fp   = X_fp
        self.X_md   = X_md
        self.graphs = graphs
        self.Y      = Y.astype(np.float32)
        n = len(Y)
        if X_fp   is not None: assert X_fp.shape[0] == n
        if X_md   is not None: assert X_md.shape[0] == n
        if graphs is not None: assert len(graphs)   == n
    def __len__(self): return len(self.Y)
    def __getitem__(self, idx):
        x_fp = torch.from_numpy(self.X_fp[idx]).float() if self.X_fp   is not None else None
        x_md = torch.from_numpy(self.X_md[idx]).float() if self.X_md   is not None else None
        g    = self.graphs[idx]                          if self.graphs is not None else None
        y    = torch.tensor(self.Y[idx], dtype=torch.float32)
        return x_fp, x_md, g, y

def collate_fn_mtl_reg(batch):
    x_fp_list, x_md_list, g_list, y_list = zip(*batch)
    has_fp = any(x is not None for x in x_fp_list)
    has_md = any(x is not None for x in x_md_list)
    has_g  = any(g is not None for g in g_list)
    x_fp_b = torch.stack(list(x_fp_list), 0) if has_fp else None
    x_md_b = torch.stack(list(x_md_list), 0) if has_md else None
    if HAS_PYG and has_g:
        valid = [g for g in g_list if g is not None]
        g_b   = Batch.from_data_list(valid) if valid else None
    else:
        g_b = None
    y_b = torch.stack(list(y_list), 0)
    return x_fp_b, x_md_b, g_b, y_b

def make_mtl_loader(X_fp, X_md, graphs, Y, batch_size=64, shuffle=True):
    ds = MTLQSARDatasetReg(X_fp, X_md, graphs, Y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      collate_fn=collate_fn_mtl_reg)

# -- Shared Encoders --
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

class GraphEncoder(nn.Module):
    '''Manual mean-pool - menghindari batch-index alignment issues.'''
    def __init__(self, in_dim, hidden_dim=64, out_dim=128, dropout=0.0):
        super().__init__()
        if not HAS_PYG: raise ImportError('torch_geometric not available')
        self.conv1   = GCNConv(in_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, batch):
        x, ei, bidx = batch.x, batch.edge_index, batch.batch
        h = F.relu(self.conv1(x, ei))
        h = self.dropout(h)
        h = F.relu(self.conv2(h, ei))
        n_graphs = int(bidx.max().item()) + 1
        out = []
        for g in range(n_graphs):
            mask = (bidx == g)
            out.append(h[mask].mean(dim=0, keepdim=True))
        return torch.cat(out, dim=0)

def _build_backbone(concat_dim, hidden_dim, n_layers, arch_type, dropout):
    arch_type = arch_type or 'uniform'
    n_layers  = max(int(n_layers), 1)
    if arch_type == 'uniform':
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    elif arch_type == 'pyramid':
        dims = [concat_dim]; cur = hidden_dim
        for _ in range(n_layers):
            dims.append(cur); cur = max(cur // 2, 32)
        layer_dims = dims
    elif arch_type == 'wide_first':
        layer_dims = ([concat_dim, hidden_dim * 2]
                      + [hidden_dim] * max(n_layers - 1, 0))
    else:
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    layers = []
    for in_d, out_d in zip(layer_dims[:-1], layer_dims[1:]):
        layers += [nn.Linear(in_d, out_d), nn.ReLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers), layer_dims[-1]

# -- MTL Model (REVISED: Task-Specific Backbone) --
class MTLModularNetReg(nn.Module):
    '''
    REVISED ARCHITECTURE:
      Shared Encoder (FP/MD/Graph)  <- transfer learning representasi molekul
          |
    +-----+-----+-----+
    BB_hERG  BB_Cav  BB_Nav   <- backbone TERPISAH per task (tidak ada gradient conflict)
       |        |       |
    Head     Head    Head     <- output pIC50 per task
    '''
    def __init__(self, in_fp=0, in_md=0, in_g=None,
                 use_fp=True, use_md=True, use_g=False,
                 hidden_dim=128, dropout=0.2,
                 n_layers=2, arch_type='uniform',
                 task_names=None):
        super().__init__()
        self.task_names      = task_names or ALL_ENDPOINTS
        self.safe_task_names = [t.replace('.', '_') for t in self.task_names]
        self.name_map        = dict(zip(self.task_names, self.safe_task_names))

        self.use_fp = use_fp and (in_fp > 0)
        self.use_md = use_md and (in_md > 0)
        self.use_g  = use_g  and (in_g is not None) and (in_g > 0)

        fp_out = md_out = g_out = 0

        # -- Shared Encoders --
        if self.use_fp:
            self.fp_encoder = MLPEncoder(in_fp, hidden_dim, hidden_dim, dropout)
            fp_out = hidden_dim
        if self.use_md:
            self.md_encoder = MLPEncoder(in_md, hidden_dim, hidden_dim, dropout)
            md_out = hidden_dim
        if self.use_g:
            self.g_encoder  = GraphEncoder(in_g, 64, hidden_dim, dropout)
            g_out  = hidden_dim

        concat_dim = fp_out + md_out + g_out
        assert concat_dim > 0, 'At least one modality must be active.'

        # -- Task-Specific Backbone + Head --
        self.task_backbones = nn.ModuleDict()
        self.task_heads     = nn.ModuleDict()

        for safe_t in self.safe_task_names:
            bb, bb_out = _build_backbone(concat_dim, hidden_dim, n_layers,
                                          arch_type, dropout)
            self.task_backbones[safe_t] = bb
            self.task_heads[safe_t] = nn.Sequential(
                nn.Linear(bb_out, bb_out // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(bb_out // 2, 1)
            )

        self.feature_dims = dict(in_fp=in_fp, in_md=in_md, in_g=in_g,
                                  concat_dim=concat_dim, hidden_dim=hidden_dim,
                                  n_layers=n_layers, arch_type=arch_type)

    def forward(self, x_fp=None, x_md=None, g_batch=None):
        feats = []
        if self.use_fp and x_fp    is not None: feats.append(self.fp_encoder(x_fp))
        if self.use_md and x_md    is not None: feats.append(self.md_encoder(x_md))
        if self.use_g  and g_batch is not None: feats.append(self.g_encoder(g_batch))
        assert feats, 'All inputs are None in forward().'
        # Shared representation
        z_shared = feats[0] if len(feats) == 1 else torch.cat(feats, dim=1)
        # Task-specific forward
        out = {}
        for orig, safe_t in self.name_map.items():
            z_task = self.task_backbones[safe_t](z_shared)
            out[orig] = self.task_heads[safe_t](z_task).squeeze(1)
        return out


## Section 5 - Loss & Evaluation

In [8]:
def mtl_loss_reg(pred_dict, y_batch, task_names, lambda_tasks=None):
    '''
    Loss sum per task dengan NaN masking.
    Union dataset -> banyak NaN -> mask wajib.
    Dibagi n_active agar skala loss stabil.
    '''
    if lambda_tasks is None:
        lambda_tasks = {t: 1.0 for t in task_names}
    criterion  = nn.SmoothL1Loss()
    total_loss = torch.tensor(0.0, device=DEVICE)
    n_active   = 0
    for j, ep in enumerate(task_names):
        y_j  = y_batch[:, j]
        mask = ~torch.isnan(y_j)
        if mask.sum() == 0: continue
        total_loss = total_loss + lambda_tasks[ep] * criterion(
            pred_dict[ep][mask], y_j[mask])
        n_active += 1
    return total_loss / max(n_active, 1)

def evaluate_mtl_reg(model, loader, task_names, target_scalers=None):
    model.eval()
    all_y  = {ep: [] for ep in task_names}
    all_p  = {ep: [] for ep in task_names}
    t_loss = 0.0; n_samp = 0
    with torch.no_grad():
        for x_fp, x_md, g_b, y_b in loader:
            y_b = y_b.to(DEVICE)
            if x_fp is not None: x_fp = x_fp.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_md is not None: x_md = x_md.to(DEVICE, non_blocking=NON_BLOCKING)
            if g_b  is not None: g_b  = g_b.to(DEVICE)
            pred = model(x_fp=x_fp, x_md=x_md, g_batch=g_b)
            loss = mtl_loss_reg(pred, y_b, task_names)
            t_loss += loss.item() * y_b.size(0); n_samp += y_b.size(0)
            for j, ep in enumerate(task_names):
                y_j  = y_b[:, j].cpu().numpy()
                mask = ~np.isnan(y_j)
                if mask.sum() == 0: continue
                p_j = pred[ep].cpu().numpy()[mask]
                y_j = y_j[mask]
                if target_scalers is not None and ep in target_scalers:
                    p_j = target_scalers[ep].inverse_transform(ep, p_j)
                    y_j = target_scalers[ep].inverse_transform(ep, y_j)
                all_y[ep].append(y_j); all_p[ep].append(p_j)
    metrics = {}
    for ep in task_names:
        if not all_y[ep]:
            metrics[ep] = dict(mse=np.nan, rmse=np.nan,
                               mae=np.nan, mape=np.nan, r2=np.nan)
            continue
        metrics[ep] = compute_regression_metrics(
            np.concatenate(all_y[ep]), np.concatenate(all_p[ep]))
    return t_loss / max(n_samp, 1), metrics

def mean_mae(metrics_dict):
    vals = [m['mae'] for m in metrics_dict.values() if not np.isnan(m['mae'])]
    return float(np.mean(vals)) if vals else 999.0

def stratified_kfold_indices(Y_mtl, n_splits=10, n_bins=10, seed=SEED):
    '''Bin mean pIC50 per compound (nanmean), lalu StratifiedKFold.'''
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mean_pic50 = np.nanmean(Y_mtl, axis=1)
    mean_pic50 = np.where(np.isnan(mean_pic50), np.nanmean(mean_pic50), mean_pic50)
    try:
        bins = pd.qcut(mean_pic50, q=n_bins, labels=False, duplicates='drop')
    except Exception:
        bins = pd.cut(mean_pic50, bins=n_bins, labels=False)
    bins = np.array(bins, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(Y_mtl)), bins))


## Section 6 - Logging & Plotting Utilities

In [9]:
def append_epoch_log(out_dir, fold_id, epoch, train_loss, val_loss,
                     mean_mae_val, metrics, lr_now):
    log_dir = os.path.join(out_dir, f'fold_{fold_id}')
    os.makedirs(log_dir, exist_ok=True)
    log_csv = os.path.join(log_dir, 'epoch_log.csv')
    file_exists = os.path.exists(log_csv)
    with open(log_csv, 'a', newline='') as f:
        w = csv.writer(f)
        if not file_exists:
            hdr = ['epoch', 'train_loss', 'val_loss', 'mean_mae', 'lr']
            for ep in metrics:
                hdr += [f'{ep}_mse', f'{ep}_rmse', f'{ep}_mae',
                        f'{ep}_mape', f'{ep}_r2']
            w.writerow(hdr)
        row = [epoch, train_loss, val_loss, mean_mae_val, lr_now]
        for ep in metrics:
            m = metrics[ep]
            row += [m['mse'], m['rmse'], m['mae'], m['mape'], m['r2']]
        w.writerow(row)

def plot_fold_curve(out_dir, fold_id, config_name):
    fold_csv = os.path.join(out_dir, f'fold_{fold_id}', 'epoch_log.csv')
    fold_png = os.path.join(out_dir, f'fold_{fold_id}', 'training_curves.png')
    if not os.path.exists(fold_csv): return
    df = pd.read_csv(fold_csv)
    if df.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val')
    axes[0].set_title(f'Loss - {config_name} fold {fold_id}')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(df['epoch'], df['mean_mae'], color='tomato')
    axes[1].set_title(f'Mean MAE - {config_name} fold {fold_id}')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()
    plt.savefig(fold_png, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'[PLOT] Saved -> {fold_png}')

def plot_final_curve(out_dir, config_name):
    log_csv = os.path.join(out_dir, 'final_training_log.csv')
    out_png = os.path.join(out_dir, 'final_training_curves.png')
    if not os.path.exists(log_csv): return
    df = pd.read_csv(log_csv)
    if df.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='steelblue')
    axes[0].set_title(f'Train Loss - {config_name} (Final)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(df['epoch'], df['lr'], label='Learning Rate', color='tomato')
    axes[1].set_title(f'LR Schedule - {config_name} (Final)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'[FINAL] Plot saved -> {out_png}')


## Section 7 - Training Loop (Single Fold)

In [ ]:
def train_one_fold_mtl(
    model, optimizer, task_names,
    train_loader, val_loader,
    target_scalers=None,
    max_epochs=MAX_EPOCHS_MTL,
    patience=PATIENCE_MTL,
    config_name='', fold_id=0, out_dir='',
    lambda_tasks=None,
):
    '''
    Training loop dengan:
    - Gradient clipping (GRAD_CLIP=5.0)
    - CosineAnnealingWarmRestarts scheduler
    - Early stopping berbasis mean MAE di original pIC50 space
    - AdamW optimizer dengan weight_decay=1e-4
    '''
    best_state = None
    best_mae   = 999.0
    no_improve = 0

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=max(max_epochs // 3, 10), T_mult=1, eta_min=1e-5)

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0; n_train = 0

        for x_fp, x_md, g_b, y_b in train_loader:
            y_b = y_b.to(DEVICE)
            if x_fp is not None: x_fp = x_fp.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_md is not None: x_md = x_md.to(DEVICE, non_blocking=NON_BLOCKING)
            if g_b  is not None: g_b  = g_b.to(DEVICE)
            optimizer.zero_grad()
            pred  = model(x_fp=x_fp, x_md=x_md, g_batch=g_b)
            loss  = mtl_loss_reg(pred, y_b, task_names, lambda_tasks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item() * y_b.size(0); n_train += y_b.size(0)

        scheduler.step()
        train_loss /= max(n_train, 1)
        val_loss, val_metrics = evaluate_mtl_reg(
            model, val_loader, task_names, target_scalers)
        val_mae = mean_mae(val_metrics)
        lr_now  = optimizer.param_groups[0]['lr']

        metric_str = ' | '.join(
            f"{ep}:MAE={val_metrics[ep]['mae']:.4f},R2={val_metrics[ep]['r2']:.4f}"
            for ep in task_names)
        print(f'[{config_name}|Fold{fold_id}] Ep{epoch:03d} '
              f'tr={train_loss:.4f} val={val_loss:.4f} '
              f'meanMAE={val_mae:.4f} lr={lr_now:.2e} | {metric_str}')

        append_epoch_log(out_dir, fold_id, epoch, train_loss,
                         val_loss, val_mae, val_metrics, lr_now)

        if val_mae < best_mae - 1e-4:
            best_mae   = val_mae
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  -> Early stopping at epoch {epoch}.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_mae


## Section 8 - Cross-Validation (Stratified K-Fold)

In [10]:
def run_cv_mtl_reg(
    mtl_data,
    use_fp=True, use_md=True, use_g=False,
    n_splits=CV_FOLDS,
    batch_size=64, lr=1e-3,
    hidden_dim=128, dropout=DROPOUT,
    n_layers=2, arch_type='uniform',
    lambda_tasks=None,
    max_epochs=MAX_EPOCHS_MTL,
    patience=PATIENCE_MTL,
    n_bins_strat=10,
):
    cfg = (['FP'] if use_fp else []) + (['MD'] if use_md else []) + (['Graph'] if use_g else [])
    config_name = '+'.join(cfg) or 'None'
    out_dir     = os.path.join(OUT_ROOT, 'MTL', config_name)
    os.makedirs(out_dir, exist_ok=True)

    X_fp    = mtl_data['X_fp']    if use_fp else None
    X_md    = mtl_data['X_md']    if use_md else None
    graphs  = mtl_data['graphs']  if use_g  else None
    Y_mtl   = mtl_data['Y_mtl']
    ep_cols = mtl_data['endpoint_cols']

    in_fp = X_fp.shape[1] if X_fp is not None else 0
    in_md = X_md.shape[1] if X_md is not None else 0
    in_g  = None
    if use_g and graphs is not None:
        g0 = next((g for g in graphs if g is not None), None)
        if g0 is not None: in_g = g0.x.shape[1]

    fold_indices = stratified_kfold_indices(Y_mtl, n_splits=n_splits,
                                             n_bins=n_bins_strat)
    fold_results = []

    for fold_id, (train_idx, val_idx) in enumerate(fold_indices, 1):
        print(f'\n===== MTL | {config_name} | Fold {fold_id}/{n_splits} '
              f'(train={len(train_idx)}, val={len(val_idx)}) =====')

        Y_tr  = Y_mtl[train_idx]; Y_val = Y_mtl[val_idx]

        # TargetScaler fit HANYA di train split
        tsc = TargetScaler()
        for j, ep in enumerate(ep_cols):
            y_tr_ep = Y_tr[:, j][~np.isnan(Y_tr[:, j])]
            if len(y_tr_ep) > 0: tsc.fit(ep, y_tr_ep)

        Y_tr_sc  = Y_tr.copy(); Y_val_sc = Y_val.copy()
        for j, ep in enumerate(ep_cols):
            if ep not in tsc.mean_: continue
            mask_tr  = ~np.isnan(Y_tr[:, j])
            mask_val = ~np.isnan(Y_val[:, j])
            Y_tr_sc[mask_tr,  j] = tsc.transform(ep, Y_tr[mask_tr,  j])
            Y_val_sc[mask_val, j] = tsc.transform(ep, Y_val[mask_val, j])

        tsc_dict = {ep: tsc for ep in ep_cols}

        X_fp_tr  = X_fp[train_idx] if X_fp is not None else None
        X_fp_val = X_fp[val_idx]   if X_fp is not None else None

        scaler_md = None
        X_md_tr = X_md_val = None
        if X_md is not None:
            scaler_md  = StandardScaler()
            X_md_tr    = scaler_md.fit_transform(X_md[train_idx])
            X_md_val   = scaler_md.transform(X_md[val_idx])

        g_tr  = [graphs[i] for i in train_idx] if graphs is not None else None
        g_val = [graphs[i] for i in val_idx]   if graphs is not None else None

        train_loader = make_mtl_loader(X_fp_tr,  X_md_tr,  g_tr,  Y_tr_sc,
                                        batch_size, shuffle=True)
        val_loader   = make_mtl_loader(X_fp_val, X_md_val, g_val, Y_val_sc,
                                        batch_size, shuffle=False)

        model = MTLModularNetReg(
            in_fp=in_fp, in_md=in_md, in_g=in_g,
            use_fp=use_fp, use_md=use_md, use_g=use_g,
            hidden_dim=hidden_dim, dropout=dropout,
            n_layers=n_layers, arch_type=arch_type,
            task_names=ep_cols,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        model, best_mae = train_one_fold_mtl(
            model, optimizer, ep_cols,
            train_loader, val_loader,
            target_scalers=tsc_dict,
            max_epochs=max_epochs, patience=patience,
            config_name=config_name, fold_id=fold_id, out_dir=out_dir,
            lambda_tasks=lambda_tasks,
        )

        fold_dir = os.path.join(out_dir, f'fold_{fold_id}')
        os.makedirs(fold_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(fold_dir, 'best_model.pt'))
        if scaler_md is not None:
            with open(os.path.join(fold_dir, 'scaler_md.pkl'), 'wb') as f:
                pickle.dump(scaler_md, f)
        with open(os.path.join(fold_dir, 'target_scaler.pkl'), 'wb') as f:
            pickle.dump(tsc, f)

        plot_fold_curve(out_dir, fold_id, config_name)
        fold_results.append({'fold': fold_id, 'best_mean_mae': float(best_mae)})

    cv_mae = float(np.mean([r['best_mean_mae'] for r in fold_results]))
    print(f'\n[CV MTL] {config_name} | CV mean MAE = {cv_mae:.4f}')
    with open(os.path.join(out_dir, 'cv_summary.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['fold', 'best_mean_mae'])
        w.writeheader(); w.writerows(fold_results)
    return fold_results, cv_mae


## Section 9 - Hyperparameter Optimisation (Optuna)

In [11]:
# =============================================================================
# SECTION 9: HYPERPARAMETER OPTIMISATION (OPTUNA) — REVISED
# Lambda weighting: auto-compute berbasis inverse frequency + korelasi
# =============================================================================

_LAMBDA_MAP = {
    'hERG':   'lambda_hERG',
    'Cav1.2': 'lambda_Cav1_2',
    'Nav1.5': 'lambda_Nav1_5',
}

def _extract_lambda(params):
    return {ep: params.get(key, 1.0) for ep, key in _LAMBDA_MAP.items()}

# ── AUTO-COMPUTE LAMBDA AWAL ──────────────────────────────────────────────────
def compute_initial_lambda(mtl_data, endpoint_order=ENDPOINT_ORDER):
    """
    Lambda awal = kombinasi:
    1. Inverse frequency  -> endpoint kecil dapat bobot lebih besar
    2. Korelasi rata-rata -> endpoint dengan korelasi tinggi dikurangi sedikit
       karena sudah banyak sinyal bersama dari task lain
    Hasil dinormalisasi agar rata-rata = 1.0
    """
    Y       = mtl_data['Y_mtl']
    ep_cols = mtl_data['endpoint_cols']

    # N valid per endpoint
    n_valid = {ep: int((~np.isnan(Y[:, j])).sum())
               for j, ep in enumerate(ep_cols)}
    print('\n[Lambda Init] N valid per endpoint:')
    for ep, n in n_valid.items():
        print(f'  {ep}: {n}')

    # Inverse frequency (dinormalisasi ke rata-rata = 1.0)
    total        = sum(n_valid.values())
    inv_freq     = {ep: total / n_valid[ep] for ep in ep_cols}
    inv_freq_sum = sum(inv_freq.values())
    inv_freq_norm = {ep: v / inv_freq_sum * len(ep_cols)
                     for ep, v in inv_freq.items()}
    print('\n[Lambda Init] Inverse-frequency lambda (sebelum korelasi):')
    for ep, v in inv_freq_norm.items():
        print(f'  {ep}: {v:.4f}')

    # Korelasi Pearson rata-rata per endpoint
    df   = pd.DataFrame(Y, columns=ep_cols)
    corr = df.corr(method='pearson', min_periods=10)
    mean_corr = {
        ep: float(corr.loc[ep, [e for e in ep_cols if e != ep]].mean())
        for ep in ep_cols
    }
    print('\n[Lambda Init] Mean Pearson correlation per endpoint:')
    for ep, v in mean_corr.items():
        print(f'  {ep}: {v:.4f}')

    # Gabungkan: korelasi tinggi -> lambda sedikit lebih kecil
    final_lambda = {
        ep: inv_freq_norm[ep] * (1.0 - mean_corr[ep] * 0.2)
        for ep in ep_cols
    }

    # Normalisasi ulang rata-rata = 1.0
    mean_lam     = sum(final_lambda.values()) / len(final_lambda)
    final_lambda = {ep: round(v / mean_lam, 4) for ep, v in final_lambda.items()}

    print('\n[Lambda Init] Final lambda (inv-freq + korelasi):')
    for ep, v in final_lambda.items():
        print(f'  {ep}: {v:.4f}')

    return final_lambda

# ── OPTUNA OBJECTIVE ──────────────────────────────────────────────────────────
def make_objective_mtl(mtl_data, use_fp, use_md, use_g):
    cfg         = (['FP'] if use_fp else []) + (['MD'] if use_md else []) + (['Graph'] if use_g else [])
    config_name = '+'.join(cfg) or 'None'

    def objective(trial: optuna.Trial):
        arch_type  = trial.suggest_categorical('arch_type',
                         ['uniform', 'pyramid', 'wide_first'])
        n_layers   = trial.suggest_int('n_layers',   2, 6)
        hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
        dropout    = trial.suggest_float('dropout',  0.0, 0.4)
        lr         = trial.suggest_float('lr',       1e-4, 5e-3, log=True)
        batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])

        # Search space proporsional terhadap INIT_LAMBDA
        # Setiap endpoint bergerak dalam range ±50% dari lambda awal
        lambda_tasks = {}
        for ep in mtl_data['endpoint_cols']:
            key  = _LAMBDA_MAP[ep]
            base = INIT_LAMBDA[ep]
            lo   = round(max(0.1, base * 0.5), 4)
            hi   = round(base * 2.0, 4)
            lambda_tasks[ep] = trial.suggest_float(key, lo, hi)

        print(f'\n[OPTUNA|{config_name}|Trial{trial.number}] '
              f'arch={arch_type} layers={n_layers} hid={hidden_dim} '
              f'lr={lr:.2e} dr={dropout:.2f} bs={batch_size} '
              f'lambda={lambda_tasks}')

        _, cv_mae = run_cv_mtl_reg(
            mtl_data=mtl_data,
            use_fp=use_fp, use_md=use_md, use_g=use_g,
            n_splits=CV_FOLDS,
            batch_size=batch_size, lr=lr,
            hidden_dim=hidden_dim, dropout=dropout,
            n_layers=n_layers, arch_type=arch_type,
            lambda_tasks=lambda_tasks,
            max_epochs=MAX_EPOCHS_MTL,
            patience=PATIENCE_MTL,
        )
        trial.set_user_attr('cv_mae', float(cv_mae))
        return float(cv_mae)
    return objective

# ── RUN OPTUNA ────────────────────────────────────────────────────────────────
def run_optuna_mtl(mtl_data, configs=None, n_trials=10):
    if configs is None:
        configs = [
            {'name': 'FP',          'use_fp': True,  'use_md': False, 'use_g': False},
            {'name': 'MD',          'use_fp': False, 'use_md': True,  'use_g': False},
            {'name': 'Graph',       'use_fp': False, 'use_md': False, 'use_g': True},
            {'name': 'FP+MD',       'use_fp': True,  'use_md': True,  'use_g': False},
            {'name': 'FP+Graph',    'use_fp': True,  'use_md': False, 'use_g': True},
            {'name': 'MD+Graph',    'use_fp': False, 'use_md': True,  'use_g': True},
            {'name': 'FP+MD+Graph', 'use_fp': True,  'use_md': True,  'use_g': True},
        ]
    master_excel = os.path.join(OUT_ROOT, 'mtl_reg_best_architectures.xlsx')
    best_results = {}

    for cfg in configs:
        cname = cfg['name']
        print(f"\n{'='*60}\n  [HPO] {cname}\n{'='*60}")
        study = optuna.create_study(study_name=f'MTL_{cname}', direction='minimize')
        study.optimize(
            make_objective_mtl(mtl_data, cfg['use_fp'],
                               cfg['use_md'], cfg['use_g']),
            n_trials=n_trials, show_progress_bar=True)

        bp = study.best_params.copy()
        bv = float(study.best_value)
        print(f'[BEST|{cname}] MAE={bv:.4f} | {bp}')

        row = {
            'Config':      cname,
            'BestCV_MAE':  round(bv, 6),
            'BestParams':  str(bp),
            'InitLambda':  str(INIT_LAMBDA),
            'NTrials':     n_trials,
            'Timestamp':   pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        df_row = pd.DataFrame([row])
        if os.path.exists(master_excel):
            df_row = pd.concat([pd.read_excel(master_excel), df_row],
                               ignore_index=True)
        df_row.to_excel(master_excel, index=False, engine='openpyxl')

        best_results[cname] = {
            'best_mae':     bv,
            'best_params':  bp,
            'lambda_tasks': _extract_lambda(bp),
        }
    return best_results

## Section 10 - Final Training on Full Dev Set

In [12]:
def train_full_mtl_reg(
    mtl_data,
    use_fp=True, use_md=True, use_g=False,
    batch_size=64, lr=1e-3,
    hidden_dim=128, dropout=DROPOUT,
    n_layers=2, arch_type='uniform',
    lambda_tasks=None,
    max_epochs=MAX_EPOCHS_MTL,
):
    cfg = (['FP'] if use_fp else []) + (['MD'] if use_md else []) + (['Graph'] if use_g else [])
    config_name = '+'.join(cfg) or 'None'
    out_dir     = os.path.join(OUT_ROOT, 'MTL', config_name, 'final')
    os.makedirs(out_dir, exist_ok=True)

    X_fp    = mtl_data['X_fp']    if use_fp else None
    X_md    = mtl_data['X_md']    if use_md else None
    graphs  = mtl_data['graphs']  if use_g  else None
    Y_mtl   = mtl_data['Y_mtl']
    ep_cols = mtl_data['endpoint_cols']
    N = len(Y_mtl)
    print(f'\n[FINAL MTL] Config={config_name} | N={N} | Eps={ep_cols}')

    if lambda_tasks is None:
        lambda_tasks = {t: 1.0 for t in ep_cols}

    # TargetScaler di full dev set
    tsc = TargetScaler()
    for j, ep in enumerate(ep_cols):
        y_ep = Y_mtl[:, j][~np.isnan(Y_mtl[:, j])]
        if len(y_ep) > 0: tsc.fit(ep, y_ep)
    Y_sc = Y_mtl.copy()
    for j, ep in enumerate(ep_cols):
        if ep not in tsc.mean_: continue
        mask = ~np.isnan(Y_mtl[:, j])
        Y_sc[mask, j] = tsc.transform(ep, Y_mtl[mask, j])

    scaler_md = None
    if X_md is not None:
        scaler_md = StandardScaler()
        X_md = scaler_md.fit_transform(X_md)

    full_loader = make_mtl_loader(X_fp, X_md, graphs, Y_sc,
                                   batch_size=batch_size, shuffle=True)

    in_fp = X_fp.shape[1] if X_fp is not None else 0
    in_md = X_md.shape[1] if X_md is not None else 0
    in_g  = None
    if use_g and graphs is not None:
        g0 = next((g for g in graphs if g is not None), None)
        if g0 is not None: in_g = g0.x.shape[1]

    model = MTLModularNetReg(
        in_fp=in_fp, in_md=in_md, in_g=in_g,
        use_fp=use_fp, use_md=use_md, use_g=use_g,
        hidden_dim=hidden_dim, dropout=dropout,
        n_layers=n_layers, arch_type=arch_type,
        task_names=ep_cols,
    ).to(DEVICE)

    n_p = sum(p.numel() for p in model.parameters())
    print(f'  Params: {n_p:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs, eta_min=lr * 0.05)

    best_state = None
    best_loss  = float('inf')

    log_csv = os.path.join(out_dir, 'final_training_log.csv')
    with open(log_csv, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'lr'])

    for epoch in range(1, max_epochs + 1):
        model.train()
        ep_loss = 0.0; n_total = 0
        for x_fp, x_md, g_b, y_b in full_loader:
            y_b = y_b.to(DEVICE)
            if x_fp is not None: x_fp = x_fp.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_md is not None: x_md = x_md.to(DEVICE, non_blocking=NON_BLOCKING)
            if g_b  is not None: g_b  = g_b.to(DEVICE)
            optimizer.zero_grad()
            pred  = model(x_fp=x_fp, x_md=x_md, g_batch=g_b)
            loss  = mtl_loss_reg(pred, y_b, ep_cols, lambda_tasks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            ep_loss += loss.item() * y_b.size(0); n_total += y_b.size(0)

        scheduler.step()
        ep_loss /= max(n_total, 1)
        lr_now   = optimizer.param_groups[0]['lr']
        print(f'[FINAL|{config_name}] Ep{epoch:03d} loss={ep_loss:.4f} lr={lr_now:.2e}')
        with open(log_csv, 'a', newline='') as f:
            csv.writer(f).writerow([epoch, ep_loss, lr_now])

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'[FINAL] Loaded best state (loss={best_loss:.4f})')

    torch.save(model.state_dict(), os.path.join(out_dir, 'final_model.pt'))
    with open(os.path.join(out_dir, 'target_scaler.pkl'), 'wb') as f:
        pickle.dump(tsc, f)
    if scaler_md is not None:
        with open(os.path.join(out_dir, 'scaler_md.pkl'), 'wb') as f:
            pickle.dump(scaler_md, f)

    arch_log = {
        'config': config_name, 'n_params': n_p,
        'hidden_dim': hidden_dim, 'dropout': dropout,
        'n_layers': n_layers, 'arch_type': arch_type,
        'lr': lr, 'batch_size': batch_size,
        'lambda_tasks': str(lambda_tasks),
        'best_train_loss': best_loss,
        'dataset': 'UNION',
        'backbone': 'task-specific',
    }
    pd.DataFrame([arch_log]).to_excel(
        os.path.join(out_dir, 'architecture.xlsx'), index=False)

    plot_final_curve(out_dir, config_name)
    print(f'[FINAL] Saved to: {out_dir}')
    return model, scaler_md, tsc


## Section 11 - Execution: HPO + Final Training

In [ ]:
# =============================================================================
# SECTION 11: EXECUTION — HPO + FINAL TRAINING [REVISED]
# =============================================================================
import ast as _ast

RUN_CONFIGS = [
    {'name': 'FP',          'use_fp': True,  'use_md': False, 'use_g': False},
    {'name': 'MD',          'use_fp': False, 'use_md': True,  'use_g': False},
    {'name': 'Graph',       'use_fp': False, 'use_md': False, 'use_g': True},
    {'name': 'FP+MD',       'use_fp': True,  'use_md': True,  'use_g': False},
    {'name': 'FP+Graph',    'use_fp': True,  'use_md': False, 'use_g': True},
    {'name': 'MD+Graph',    'use_fp': False, 'use_md': True,  'use_g': True},
    {'name': 'FP+MD+Graph', 'use_fp': True,  'use_md': True,  'use_g': True},
]

N_TRIALS       = 10
RUN_HPO        = True
RUN_FULL_TRAIN = True

seed_everything(SEED)
best_hp_all  = {}
final_models = {}

# ── PRE-COMPUTE LAMBDA AWAL ───────────────────────────────────────────────────
# Wajib dijalankan sebelum HPO agar search space Optuna proporsional
INIT_LAMBDA = compute_initial_lambda(MTL_DATA)
print('\n[INFO] INIT_LAMBDA yang akan dipakai sebagai basis search space:')
for ep, v in INIT_LAMBDA.items():
    print(f'  {ep}: {v:.4f}  '
          f'(search range: [{round(max(0.1, v*0.5), 4)}, {round(v*2.0, 4)}])')

# ── STAGE 1: HPO ──────────────────────────────────────────────────────────────
if RUN_HPO:
    print('\n' + '='*60)
    print('  STAGE 1: HYPERPARAMETER OPTIMISATION (OPTUNA)')
    print('='*60)
    best_hp_all = run_optuna_mtl(
        mtl_data=MTL_DATA, configs=RUN_CONFIGS, n_trials=N_TRIALS)
    print('\n[DONE] HPO finished.')
    for cfg_name, hp in best_hp_all.items():
        print(f'  {cfg_name}: MAE={hp["best_mae"]:.4f} | '
              f'lambda={hp["lambda_tasks"]}')
else:
    master_excel = os.path.join(OUT_ROOT, 'mtl_reg_best_architectures.xlsx')
    if os.path.exists(master_excel):
        df_hp = pd.read_excel(master_excel, engine='openpyxl')
        for _, row in df_hp.iterrows():
            cname      = row['Config']
            raw_params = _ast.literal_eval(row['BestParams'])
            best_hp_all[cname] = {
                'best_mae':     row['BestCV_MAE'],
                'best_params':  raw_params,
                'lambda_tasks': _extract_lambda(raw_params),
            }
        print(f'[LOAD] {len(best_hp_all)} configs loaded from {master_excel}')
    else:
        print(f'[WARN] No HPO file found at {master_excel}. best_hp_all is empty.')

# ── STAGE 2: FINAL TRAINING ───────────────────────────────────────────────────
if RUN_FULL_TRAIN and best_hp_all:
    print('\n' + '='*60)
    print('  STAGE 2: FINAL TRAINING (FULL DEV SET)')
    print('='*60)
    for cfg in RUN_CONFIGS:
        cname = cfg['name']
        if cname not in best_hp_all:
            print(f'[SKIP] {cname} not in best_hp_all'); continue
        bp = best_hp_all[cname]['best_params']
        lt = best_hp_all[cname]['lambda_tasks']
        print(f'\n[FINAL] {cname} | HP={bp} | lambda={lt}')
        model, scaler_md, tsc = train_full_mtl_reg(
            mtl_data=MTL_DATA,
            use_fp=cfg['use_fp'], use_md=cfg['use_md'], use_g=cfg['use_g'],
            batch_size=bp.get('batch_size', 64),
            lr=bp['lr'],
            hidden_dim=bp['hidden_dim'],
            dropout=bp['dropout'],
            n_layers=bp['n_layers'],
            arch_type=bp['arch_type'],
            lambda_tasks=lt,
            max_epochs=MAX_EPOCHS_MTL,
        )
        final_models[cname] = {'model': model, 'scaler_md': scaler_md, 'tsc': tsc}
    print('\n[DONE] Final training complete.')
    for k in final_models: print(f'  OK {k}')
elif RUN_FULL_TRAIN and not best_hp_all:
    print('[SKIP] RUN_FULL_TRAIN=True but best_hp_all is empty.')
else:
    print('[SKIP] RUN_FULL_TRAIN=False.')

In [14]:
import pandas as pd
import numpy as np

# Ambil pIC50 per endpoint dari MTL_DATA (union)
Y = MTL_DATA['Y_mtl']
ep_cols = MTL_DATA['endpoint_cols']

# Buat DataFrame — hanya baris yang punya minimal 2 endpoint (untuk korelasi valid)
df_corr = pd.DataFrame(Y, columns=ep_cols)

print("=" * 50)
print("Jumlah senyawa per endpoint:")
for ep in ep_cols:
    print(f"  {ep}: {df_corr[ep].notna().sum()} senyawa")

print("\nJumlah senyawa overlap antar endpoint:")
for i, ep1 in enumerate(ep_cols):
    for ep2 in ep_cols[i+1:]:
        both = df_corr[[ep1, ep2]].dropna()
        print(f"  {ep1} & {ep2}: {len(both)} senyawa overlap")

print("\nKorelasi Pearson (hanya senyawa yang overlap):")
corr_matrix = df_corr.corr(method='pearson', min_periods=10)
print(corr_matrix.round(4))

print("\nKorelasi Spearman (lebih robust, non-linear):")
corr_spearman = df_corr.corr(method='spearman', min_periods=10)
print(corr_spearman.round(4))

print("\nInterpretasi:")
for i, ep1 in enumerate(ep_cols):
    for ep2 in ep_cols[i+1:]:
        r = corr_matrix.loc[ep1, ep2]
        if abs(r) >= 0.7:
            strength = "KUAT -> MTL sangat dianjurkan"
        elif abs(r) >= 0.4:
            strength = "SEDANG -> MTL mungkin membantu"
        elif abs(r) >= 0.2:
            strength = "LEMAH -> MTL mungkin tidak membantu"
        else:
            strength = "SANGAT LEMAH -> STL lebih disarankan"
        print(f"  {ep1} vs {ep2}: r={r:.4f} -> {strength}")

Jumlah senyawa per endpoint:
  hERG: 20020 senyawa
  Cav1.2: 1138 senyawa
  Nav1.5: 4599 senyawa

Jumlah senyawa overlap antar endpoint:
  hERG & Cav1.2: 263 senyawa overlap
  hERG & Nav1.5: 472 senyawa overlap
  Cav1.2 & Nav1.5: 305 senyawa overlap

Korelasi Pearson (hanya senyawa yang overlap):
          hERG  Cav1.2  Nav1.5
hERG    1.0000  0.3145  0.5997
Cav1.2  0.3145  1.0000  0.5636
Nav1.5  0.5997  0.5636  1.0000

Korelasi Spearman (lebih robust, non-linear):
          hERG  Cav1.2  Nav1.5
hERG    1.0000  0.3907  0.6516
Cav1.2  0.3907  1.0000  0.5374
Nav1.5  0.6516  0.5374  1.0000

Interpretasi:
  hERG vs Cav1.2: r=0.3145 -> LEMAH -> MTL mungkin tidak membantu
  hERG vs Nav1.5: r=0.5997 -> SEDANG -> MTL mungkin membantu
  Cav1.2 vs Nav1.5: r=0.5636 -> SEDANG -> MTL mungkin membantu


In [13]:
# Setelah HPO selesai, print best lambda per config
for cname, hp in best_hp_all.items():
    lt = hp['lambda_tasks']
    print(f'{cname}: lambda={lt} | MAE={hp["best_mae"]:.4f}')

NameError: name 'best_hp_all' is not defined

## Section 12 - Manual Override (Bypass HPO)

In [15]:
#=============================================================================
# SECTION 12: MANUAL OVERRIDE (bypass HPO)
#=============================================================================
# Manual training dengan hyperparameter terbaik (bypass Hyperparameter Optimization)

MANUAL_HP = {
    
    # ==================== FP Only ====================
    "FP": {
        "use_fp": True, 
        "use_md": False, 
        "use_g": False,
        "arch_type": "uniform", 
        "n_layers": 3, 
        "hidden_dim": 511,
        "dropout": 0.012787584591142764, 
        "lr": 0.0003155246997505474, 
        "batch_size": 64,
        "lambda_tasks": {
            "hERG": 0.131,
            "Cav1.2": 2.3142,
            "Nav1.5": 0.5547
        },
    },

    # ==================== MD Only ====================
    "MD": {
        "use_fp": False, 
        "use_md": True, 
        "use_g": False,
        "arch_type": "wide_first", 
        "n_layers": 3, 
        "hidden_dim": 242,
        "dropout": 0.07051747878721783, 
        "lr": 0.0003398281730728331, 
        "batch_size": 64,
        "lambda_tasks": {
            "hERG": 0.20993303878304975,
            "Cav1.2": 1.8407834685922155,
            "Nav1.5": 0.56440152361426
        },
    },
}

# ==================== Run Full Training for Selected Configs ====================
print("=== MANUAL OVERRIDE: Starting Full Training (bypass HPO) ===\n")

for cname, hp in MANUAL_HP.items():
    print(f"▶ Starting full MTL Regression training for: {cname}")
    
    train_full_mtl_reg(
        mtl_data=MTL_DATA,
        use_fp=hp["use_fp"], 
        use_md=hp["use_md"], 
        use_g=hp["use_g"],
        batch_size=hp["batch_size"], 
        lr=hp["lr"],
        hidden_dim=hp["hidden_dim"], 
        dropout=hp["dropout"],
        n_layers=hp["n_layers"], 
        arch_type=hp["arch_type"],
        lambda_tasks=hp["lambda_tasks"],
        max_epochs=MAX_EPOCHS_MTL,
    )
    
    print(f"✔ Finished training for: {cname}\n")

print("=== All manual training configurations completed ===")

=== MANUAL OVERRIDE: Starting Full Training (bypass HPO) ===

▶ Starting full MTL Regression training for: FP

[FINAL MTL] Config=FP | N=24880 | Eps=['hERG', 'Cav1.2', 'Nav1.5']
  Params: 3,617,880
[FINAL|FP] Ep001 loss=0.2076 lr=3.15e-04
[FINAL|FP] Ep002 loss=0.1201 lr=3.15e-04
[FINAL|FP] Ep003 loss=0.0882 lr=3.14e-04
[FINAL|FP] Ep004 loss=0.0732 lr=3.13e-04
[FINAL|FP] Ep005 loss=0.0591 lr=3.12e-04
[FINAL|FP] Ep006 loss=0.0529 lr=3.11e-04
[FINAL|FP] Ep007 loss=0.0455 lr=3.09e-04
[FINAL|FP] Ep008 loss=0.0381 lr=3.07e-04
[FINAL|FP] Ep009 loss=0.0421 lr=3.05e-04
[FINAL|FP] Ep010 loss=0.0370 lr=3.03e-04
[FINAL|FP] Ep011 loss=0.0303 lr=3.00e-04
[FINAL|FP] Ep012 loss=0.0303 lr=2.97e-04
[FINAL|FP] Ep013 loss=0.0295 lr=2.94e-04
[FINAL|FP] Ep014 loss=0.0228 lr=2.90e-04
[FINAL|FP] Ep015 loss=0.0258 lr=2.87e-04
[FINAL|FP] Ep016 loss=0.0330 lr=2.83e-04
[FINAL|FP] Ep017 loss=0.0283 lr=2.79e-04
[FINAL|FP] Ep018 loss=0.0241 lr=2.75e-04
[FINAL|FP] Ep019 loss=0.0258 lr=2.71e-04
[FINAL|FP] Ep020 loss=0